In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('defect-vision-industrial-defect-detection-challenge')

print("Path to competition files:", path)

In [ ]:
import os, glob, random, math, time, gc, warnings, urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from PIL import Image, ImageFile, ImageDraw, ImageFilter
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import models
from scipy.ndimage import gaussian_filter
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

class CFG:
    SEED = 42
    IMG_SIZE = 256                  # try 224 / 320
    BACKBONES = ["wrn50"]           # ensemble: ["wrn50", "resnet50", "resnet18"]
    BACKBONE_WEIGHTS = {}           # offline: {"wrn50": "/kaggle/input/<dataset>/<wide_resnet50_2 V1 .pth>"}
    FEAT_DIM = 1024                 # total patch feature dim (split across layer2/layer3)
    NEIGHBORHOOD = 3                # local avg-pool window on feature maps
    CORESET_RATIO = None            # None -> 1.0 on GPU, 0.05 on CPU
    BANK_ROT = {"bottle": (0,), "cable": (0,), "hazelnut": (0,), "screw": (0,)}
                                    # add 90/180/270 for rotation-invariant categories
                                    # (e.g. hazelnut: (0,90,180,270)); do NOT for cable (layout matters)
    DENOISE_SIGMA = 0.0             # >0 applies Gaussian blur to ALL images (train+test); test 0.6-1.0
    AGG = "max_smooth"              # "max" | "max_smooth" | "topk"
    SMOOTH_SIGMA = 1.0
    TOPK_FRAC = 0.01
    THRESH_MODE = "f1_estimate"     # "f1_estimate" | "quantile" | "fixed_z"
    NORMAL_Q = 0.99                 # for "quantile" mode / fallback
    FIXED_Z = 3.0                   # for "fixed_z" mode
    MIN_NORMAL_Q = 0.80             # threshold never below this normal quantile
    STOREY_LAMBDA = 0.5
    N_BOOT = 300
    MIN_EST_DEFECTS = 3
    BATCH = 16
    BANK_CHUNK = 65536

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG.SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if CFG.CORESET_RATIO is None:
    CFG.CORESET_RATIO = 1.0 if DEVICE.type == "cuda" else 0.05
print("torch", torch.__version__, "| device:", DEVICE, "| coreset ratio:", CFG.CORESET_RATIO)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: no GPU. Expect slow scoring; consider IMG_SIZE=224.")

try:
    urllib.request.urlopen("https://download.pytorch.org/models/", timeout=5); INTERNET = True
except urllib.error.HTTPError:
    INTERNET = True
except Exception:
    INTERNET = False
print("Internet reachable (needed for pretrained weights unless BACKBONE_WEIGHTS is set):", INTERNET)

In [ ]:
TRAIN_CSV = None
for p in glob.glob("/kaggle/input/**/train.csv", recursive=True):
    if os.path.exists(os.path.join(os.path.dirname(p), "test.csv")):
        TRAIN_CSV = p; break
if TRAIN_CSV is None:
    raise FileNotFoundError("Add the competition dataset to this notebook (Add Input).")
DATA_DIR = os.path.dirname(TRAIN_CSV)
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
SS = os.path.join(DATA_DIR, "sample_submission.csv")
if not os.path.exists(SS):
    SS = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)[0]
sample_sub = pd.read_csv(SS)

FILE_INDEX, STEM_INDEX = {}, {}
for root, _, files in os.walk(DATA_DIR):
    for f in files:
        full = os.path.join(root, f)
        FILE_INDEX.setdefault(f, full); STEM_INDEX.setdefault(os.path.splitext(f)[0], full)

def resolve_path(p):
    p = str(p); c = os.path.join(DATA_DIR, p)
    if os.path.isfile(c): return c
    b = os.path.basename(p)
    return FILE_INDEX.get(b) or STEM_INDEX.get(os.path.splitext(b)[0])

for df in (train_df, test_df):
    df["full_path"] = df["path"].map(resolve_path)
    miss = df[df["full_path"].isna()]
    if len(miss): raise FileNotFoundError(f"{len(miss)} images not found, e.g. {miss['path'].head().tolist()}")

assert train_df["id"].is_unique and test_df["id"].is_unique
print("Train labels:", train_df["label"].value_counts().to_dict(), "(expect only 'good')")
print("Train:", train_df.shape, "| Test:", test_df.shape)
print(pd.concat([train_df.category.value_counts().rename("train"),
                 test_df.category.value_counts().rename("test")], axis=1))
print("\nsample_submission columns:", sample_sub.columns.tolist(), "| rows:", len(sample_sub))
print(sample_sub.head(3))
print("sample label values:", sample_sub.iloc[:, 1].unique()[:5])
assert set(sample_sub.iloc[:, 0]) == set(test_df["id"]), "sample_submission ids differ from test.csv ids"

In [ ]:
import time, os, numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image, ImageFile, ImageFilter
S = CFG.IMG_SIZE

def noise_sigma(g):
    c = (g[:-2,:-2] - 2*g[:-2,1:-1] + g[:-2,2:] - 2*g[1:-1,:-2] + 4*g[1:-1,1:-1]
         - 2*g[1:-1,2:] + g[2:,:-2] - 2*g[2:,1:-1] + g[2:,2:])
    return float(np.sqrt(np.pi/2) / 6 * np.abs(c).mean())

def read_one(path):
    t = time.time()
    try:
        im = Image.open(path); im.load(); trunc = False
    except Exception as e:
        raise RuntimeError(f"Unreadable image: {path} ({e})")   # strict: no silent partial images
    fmt, mode, (w, h) = im.format, im.mode, im.size
    rgb = im.convert("RGB")
    ai = np.asarray(rgb).astype(np.int16)
    gray = (np.abs(ai[..., 0]-ai[..., 1]).mean() + np.abs(ai[..., 1]-ai[..., 2]).mean()) < 1.0
    noise = noise_sigma(ai.mean(2).astype(np.float32))
    r = rgb.resize((S, S), Image.BICUBIC)
    if CFG.DENOISE_SIGMA > 0:
        r = r.filter(ImageFilter.GaussianBlur(CFG.DENOISE_SIGMA))
    meta = dict(orig_w=w, orig_h=h, fmt=fmt, mode=mode, gray=bool(gray), noise=noise,
                kb=os.path.getsize(path)/1024, truncated=trunc, read_s=time.time()-t)
    return np.asarray(r, dtype=np.uint8).copy(), meta

def load_split(df, name, workers=16):
    cache_a, cache_m = f"/kaggle/working/{name}_{S}_{CFG.DENOISE_SIGMA}.npy", f"/kaggle/working/{name}_meta.pkl"
    if os.path.exists(cache_a) and os.path.exists(cache_m):
        print(f"{name}: loaded from cache"); return np.load(cache_a), pd.read_pickle(cache_m)
    paths = df["full_path"].tolist(); n = len(paths)
    arr = np.zeros((n, S, S, 3), np.uint8); metas = [None]*n; t0 = time.time()
    with ThreadPoolExecutor(workers) as ex:
        futs = {ex.submit(read_one, p): i for i, p in enumerate(paths)}
        for k, f in enumerate(as_completed(futs), 1):
            i = futs[f]; arr[i], metas[i] = f.result()
            if k % 50 == 0 or k == n:
                print(f"  {name}: {k}/{n} ({time.time()-t0:.0f}s)", flush=True)
    meta = pd.concat([df.reset_index(drop=True), pd.DataFrame(metas)], axis=1)
    np.save(cache_a, arr); meta.to_pickle(cache_m)
    return arr, meta

# quick I/O diagnostic: serial read time for 10 test files
t = time.time()
for p in test_df["full_path"].head(10): Image.open(p).load()
print(f"10 test images, serial: {time.time()-t:.1f}s")

t0 = time.time()
if "train_arr" in globals() and "train_meta" in globals() and train_arr.shape[1] == S:
    print("train already in memory; reusing")
else:
    train_arr, train_meta = load_split(train_df, "train")
test_arr, test_meta = load_split(test_df, "test")
print(f"Done in {time.time()-t0:.0f}s | train {train_arr.shape} | test {test_arr.shape}")
print("slowest reads (s):", test_meta.read_s.nlargest(3).round(1).tolist())

both = pd.concat([train_meta.assign(split="train"), test_meta.assign(split="test")])
print("\nOriginal size share by split:\n", pd.crosstab(both.split, both.orig_w, normalize="index").round(2))
print("\nFormat share by split:\n", pd.crosstab(both.split, both.fmt, normalize="index").round(2))
print("\nGrayscale share by split/category:\n", both.groupby(["split", "category"]).gray.mean().unstack().round(2))
print("\nNoise sigma quantiles by split:\n", both.groupby("split").noise.quantile([.1, .5, .9, .99]).unstack().round(2))

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
SPEC = {
    "wrn50":    (models.wide_resnet50_2, models.Wide_ResNet50_2_Weights.IMAGENET1K_V1),
    "resnet50": (models.resnet50,        models.ResNet50_Weights.IMAGENET1K_V1),
    "resnet18": (models.resnet18,        models.ResNet18_Weights.IMAGENET1K_V1),
}

def load_backbone(name):
    ctor, w = SPEC[name]
    local = CFG.BACKBONE_WEIGHTS.get(name)
    if local:
        m = ctor(weights=None); m.load_state_dict(torch.load(local, map_location="cpu"), strict=True)
        print(f"{name}: loaded local weights {local}"); return m
    try:
        m = ctor(weights=w); print(f"{name}: pretrained ImageNet weights loaded"); return m
    except Exception as e:
        raise RuntimeError(
            f"Could not download pretrained weights for '{name}' ({str(e)[:150]}).\n"
            "Fix: turn Internet ON in notebook settings, or attach a Kaggle dataset containing the "
            "torchvision ImageNet weights file and set CFG.BACKBONE_WEIGHTS. "
            "Refusing to continue with random weights.")

class PatchFeat(nn.Module):
    def __init__(self, bb, feat_dim, nb):
        super().__init__()
        bb.eval()
        self.stem = nn.Sequential(bb.conv1, bb.bn1, bb.relu, bb.maxpool)
        self.l1, self.l2, self.l3 = bb.layer1, bb.layer2, bb.layer3
        self.nb = nb
        with torch.no_grad():
            x = self.l1(self.stem(torch.zeros(1, 3, 64, 64))); f2 = self.l2(x); f3 = self.l3(f2)
        self.d2, self.d3 = min(feat_dim // 2, f2.shape[1]), min(feat_dim // 2, f3.shape[1])
        self.out_dim = self.d2 + self.d3

    @staticmethod
    def _cpool(f, d):
        B, C, H, W = f.shape
        f = F.adaptive_avg_pool1d(f.permute(0, 2, 3, 1).reshape(B*H*W, 1, C), d)
        return f.reshape(B, H, W, d)

    def forward(self, x):
        x = self.l1(self.stem(x)); f2 = self.l2(x); f3 = self.l3(f2)
        f2 = F.avg_pool2d(f2, self.nb, 1, self.nb // 2)
        f3 = F.avg_pool2d(f3, self.nb, 1, self.nb // 2)
        f3 = F.interpolate(f3, size=f2.shape[-2:], mode="bilinear", align_corners=False)
        return torch.cat([self._cpool(f2, self.d2), self._cpool(f3, self.d3)], dim=-1)  # B,H,W,D

@torch.no_grad()
def extract(model, imgs, bs=None):
    bs = bs or CFG.BATCH; out = []
    for i in range(0, len(imgs), bs):
        x = torch.from_numpy(imgs[i:i+bs]).to(DEVICE).permute(0, 3, 1, 2).float().div_(255)
        out.append(model((x - MEAN) / STD).half().cpu())
    return torch.cat(out)   # (N,H,W,D) float16 on CPU
print("Feature extractor ready.")

In [ ]:
@torch.no_grad()
def greedy_coreset(X, n, proj_dim=128, seed=0):
    g = torch.Generator().manual_seed(seed)
    R = (torch.randn(X.shape[1], proj_dim, generator=g) / math.sqrt(proj_dim)).to(X.device)
    Z = X @ R; N = Z.shape[0]
    sel = [int(torch.randint(N, (1,), generator=g))]
    mind = ((Z - Z[sel[0]])**2).sum(1)
    for _ in range(n - 1):
        j = int(torch.argmax(mind)); sel.append(j)
        mind = torch.minimum(mind, ((Z - Z[j])**2).sum(1))
    return torch.tensor(sel, device=X.device)

class PatchBank:
    def fit(self, feats, src, coreset_ratio):
        M, H, W, D = feats.shape; self.H, self.W, self.D = H, W, D
        X = feats.reshape(-1, D).to(DEVICE).float()
        s = torch.from_numpy(np.repeat(src, H*W)).to(DEVICE)
        if coreset_ratio < 1.0:
            idx = greedy_coreset(X, max(1000, int(len(X) * coreset_ratio))); X, s = X[idx], s[idx]
        self.X, self.src, self.sq = X, s, (X*X).sum(1)
        return self

    @torch.no_grad()
    def score_maps(self, feats, exclude=None):
        """Patch anomaly maps = distance to nearest bank patch. exclude[i] = image id whose patches
        are masked out of the bank for query i (leave-one-out)."""
        n = feats.shape[0]; out = np.zeros((n, self.H, self.W), np.float32)
        for i in range(n):
            q = feats[i].reshape(-1, self.D).to(DEVICE).float()
            qsq = (q*q).sum(1, keepdim=True)
            best = torch.full((q.shape[0],), float("inf"), device=DEVICE)
            for s0 in range(0, len(self.X), CFG.BANK_CHUNK):
                s1 = s0 + CFG.BANK_CHUNK
                d = qsq + self.sq[s0:s1][None, :] - 2.0 * (q @ self.X[s0:s1].T)
                if exclude is not None:
                    d.masked_fill_((self.src[s0:s1] == int(exclude[i]))[None, :], float("inf"))
                best = torch.minimum(best, d.min(1).values)
            out[i] = best.clamp_min(0).sqrt().reshape(self.H, self.W).cpu().numpy()
        return out

AGGS = ["max", "max_smooth", "topk"]
def aggregate(maps):
    flat = maps.reshape(len(maps), -1)
    k = max(1, int(round(CFG.TOPK_FRAC * flat.shape[1])))
    return {"max": flat.max(1),
            "max_smooth": np.array([gaussian_filter(m, CFG.SMOOTH_SIGMA).max() for m in maps]),
            "topk": np.sort(flat, axis=1)[:, -k:].mean(1)}
print("Bank + scoring ready.")

In [ ]:
def make_synthetic(img, rng):
    h, w, _ = img.shape; out = img.copy()
    isgray = np.abs(img[..., 0].astype(int) - img[..., 1]).mean() < 1.0
    kind = str(rng.choice(["cutpaste", "scratch", "blob"]))
    if kind == "cutpaste":
        ph, pw = int(rng.uniform(.06, .22) * h), int(rng.uniform(.06, .22) * w)
        sy, sx = rng.integers(0, h - ph), rng.integers(0, w - pw)
        dy, dx = rng.integers(0, h - ph), rng.integers(0, w - pw)
        p = img[sy:sy+ph, sx:sx+pw].astype(np.float32)
        out[dy:dy+ph, dx:dx+pw] = np.clip(p * rng.uniform(.75, 1.25) + rng.uniform(-20, 20), 0, 255).astype(np.uint8)
    elif kind == "scratch":
        pil = Image.fromarray(out); d = ImageDraw.Draw(pil)
        L, ang = rng.uniform(.15, .5) * h, rng.uniform(0, np.pi)
        cx, cy = rng.uniform(.2, .8) * w, rng.uniform(.2, .8) * h
        col = int(rng.integers(0, 60)) if rng.random() < .5 else int(rng.integers(200, 256))
        d.line([(cx - L/2*np.cos(ang), cy - L/2*np.sin(ang)), (cx + L/2*np.cos(ang), cy + L/2*np.sin(ang))],
               fill=(col, col, col), width=int(rng.integers(1, 4)))
        out = np.asarray(pil).copy()
    else:
        ph, pw = int(rng.uniform(.05, .14) * h), int(rng.uniform(.05, .14) * w)
        y0, x0 = rng.integers(0, h - ph), rng.integers(0, w - pw)
        yy, xx = np.mgrid[0:ph, 0:pw]
        mask = (((yy - ph/2)/(ph/2))**2 + ((xx - pw/2)/(pw/2))**2) <= 1
        color = np.repeat(rng.integers(0, 256), 3) if isgray else rng.integers(0, 256, 3)
        reg = out[y0:y0+ph, x0:x0+pw].astype(np.float32); a = rng.uniform(.4, .8)
        reg[mask] = (1 - a) * reg[mask] + a * color
        out[y0:y0+ph, x0:x0+pw] = reg.astype(np.uint8)
    return out

CATS = sorted(train_meta.category.unique())
tr_idx = {c: np.where(train_meta.category.values == c)[0] for c in CATS}
te_idx = {c: np.where(test_meta.category.values == c)[0] for c in CATS}
assert set(test_meta.category) <= set(CATS)

SYN_ARR = {}
for ci, cat in enumerate(CATS):
    rng_s = np.random.default_rng(CFG.SEED + ci)
    SYN_ARR[cat] = np.stack([make_synthetic(train_arr[i], rng_s) for i in tr_idx[cat]])

fig, ax = plt.subplots(len(CATS), 4, figsize=(10, 2.6 * len(CATS)))
for r, cat in enumerate(CATS):
    for c in range(4):
        ax[r, c].imshow(SYN_ARR[cat][c]); ax[r, c].axis("off")
    ax[r, 0].set_title(f"{cat}: synthetic examples", fontsize=8, loc="left")
plt.tight_layout(); plt.show()

In [ ]:
# FAST PATCHCORE-STYLE FEATURE EXTRACTION AND SCORING
# Run setup cells 0-6 before this cell.

import time
import gc
import numpy as np
import torch

# ---------------- CONFIGURATION ----------------
# Use the backbone names defined in CFG (e.g. "wrn50"), not torchvision constructor names.
BACKBONES = [CFG.BACKBONES[0]]
CORESET_RATIO = 0.05
USE_ROTATIONS = False

RES = {c: {} for c in CATS}

print("Device:", DEVICE)
print("Backbones:", BACKBONES)
print("Coreset ratio:", CORESET_RATIO)
print("Rotations enabled:", USE_ROTATIONS)

# ---------------- FEATURE EXTRACTION ----------------
for bb in BACKBONES:
    print(f"\nLoading backbone: {bb}", flush=True)

    model = PatchFeat(
        load_backbone(bb),
        CFG.FEAT_DIM,
        CFG.NEIGHBORHOOD
    ).to(DEVICE).eval()

    for cat in CATS:
        t0 = time.time()
        print(f"\nProcessing {cat}...", flush=True)

        X = train_arr[tr_idx[cat]]
        X_test = test_arr[te_idx[cat]]
        X_syn = SYN_ARR[cat]
        n = len(X)

        if n == 0:
            raise ValueError(f"No normal training images for category: {cat}")

        # Extract features for normal training images.
        with torch.inference_mode():
            f0 = extract(model, X)

        bank_features = [f0]
        bank_sources = [np.arange(n)]

        # Optional rotation augmentation (disabled by default).
        if USE_ROTATIONS:
            for rot in CFG.BANK_ROT.get(cat, (0,)):
                if rot % 360:
                    Xr = np.ascontiguousarray(
                        np.rot90(X, rot // 90, axes=(1, 2))
                    )
                    with torch.inference_mode():
                        fr = extract(model, Xr)
                    bank_features.append(fr)
                    bank_sources.append(np.arange(n))
                    del fr, Xr

        # Build reference memory bank.
        bank = PatchBank().fit(
            torch.cat(bank_features, dim=0),
            np.concatenate(bank_sources),
            CORESET_RATIO
        )
        del bank_features, bank_sources
        gc.collect()

        # Keep aggregate() output as a dict keyed by "max", "max_smooth", and "topk".
        # Downstream cells expect this structure: RES[cat][bb]["loo"][agg].
        loo_maps = bank.score_maps(f0, exclude=np.arange(n))
        loo_scores = aggregate(loo_maps)
        del loo_maps

        # Score synthetic anomalies. The synthetic images correspond one-to-one
        # with the normal training images, so use the same source-image exclusions.
        with torch.inference_mode():
            syn_features = extract(model, X_syn)
        syn_maps = bank.score_maps(syn_features, exclude=np.arange(n))
        syn_scores = aggregate(syn_maps)
        del syn_features, syn_maps
        gc.collect()

        # Score test images.
        with torch.inference_mode():
            test_features = extract(model, X_test)
        test_maps = bank.score_maps(test_features)
        test_scores = aggregate(test_maps)
        del test_maps

        RES[cat][bb] = {
            "loo": loo_scores,
            "syn": syn_scores,
            "test": test_scores,
        }

        print(
            f"[{bb}] {cat}: "
            f"train={n}, "
            f"bank={len(bank.X):,} patches, "
            f"test={len(te_idx[cat])}, "
            f"time={time.time() - t0:.1f}s",
            flush=True
        )

        del bank, f0, loo_scores, syn_scores, test_features, test_scores
        gc.collect()

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    del model
    gc.collect()

print("\nScoring complete.")


In [ ]:
def ensemble_scores(agg=None):
    agg = agg or CFG.AGG; E = {}
    for cat in CATS:
        parts = {"loo": [], "syn": [], "test": []}
        for bb in CFG.BACKBONES:
            r = RES[cat][bb]; mu, sd = r["loo"][agg].mean(), r["loo"][agg].std(ddof=1) + 1e-8
            for k in parts: parts[k].append((r[k][agg] - mu) / sd)   # z relative to NORMAL LOO scores
        E[cat] = {k: np.mean(v, 0) for k, v in parts.items()}
    return E
E = ensemble_scores()

# synthetic AUROC for each backbone/aggregation (relative comparison only)
rows = []
for cat in CATS:
    for bb in CFG.BACKBONES:
        for agg in AGGS:
            r = RES[cat][bb]; y = np.r_[np.zeros(len(r["loo"][agg])), np.ones(len(r["syn"][agg]))]
            rows.append(dict(cat=cat, backbone=bb, agg=agg, syn_auroc=roc_auc_score(y, np.r_[r["loo"][agg], r["syn"][agg]])))
print("Synthetic-defect AUROC (NOT real-defect performance):")
print(pd.DataFrame(rows).pivot_table(index=["cat", "backbone"], columns="agg", values="syn_auroc").round(3))

# nuisance sensitivity on NORMAL images (any trend here is false-positive risk, not defects)
print("\nNuisance check on normal LOO z-scores:")
for cat in CATS:
    m = train_meta.iloc[tr_idx[cat]].assign(z=E[cat]["loo"])
    print(f"{cat}: rho(noise,z)={spearmanr(m.noise, m.z)[0]:.2f} | by fmt {m.groupby('fmt').z.mean().round(2).to_dict()}"
          f" | by gray {m.groupby('gray').z.mean().round(2).to_dict()} | by size {m.groupby('orig_w').z.mean().round(2).to_dict()}")

fig, ax = plt.subplots(1, len(CATS), figsize=(4.2 * len(CATS), 3))
for a, cat in zip(np.atleast_1d(ax), CATS):
    a.hist(E[cat]["loo"], 25, alpha=.5, density=True, label="normal (LOO)")
    a.hist(E[cat]["syn"], 25, alpha=.4, density=True, label="synthetic")
    a.hist(E[cat]["test"], 25, alpha=.4, density=True, label="TEST")
    a.set_title(cat); a.set_xlabel("z vs normal")
np.atleast_1d(ax)[0].legend(fontsize=7); plt.tight_layout(); plt.show()

In [ ]:
rng_b = np.random.default_rng(CFG.SEED)

def curve(ts, ns, lam):
    ns = np.sort(ns); n, m = len(ns), len(ts)
    p = (1 + n - np.searchsorted(ns, ts, side="left")) / (n + 1)        # empirical p-value vs normals
    pi0 = min(1.0, float(np.mean(p > lam)) / (1 - lam))                  # Storey: estimated fraction of normals
    cand = np.unique(ts); cand = cand[cand >= np.quantile(ns, CFG.MIN_NORMAL_Q)]
    if len(cand) == 0: return None
    pred = m - np.searchsorted(np.sort(ts), cand, side="left")
    fpr = (1 + n - np.searchsorted(ns, cand, side="left")) / (n + 1)
    D = (1 - pi0) * m; fp = pi0 * m * fpr
    tp = np.clip(pred - fp, 0, D)
    return dict(cand=cand, f1=2 * tp / np.maximum(pred + D, 1e-9), pi0=pi0, D=D)

def best_t(c):
    near = c["f1"] >= c["f1"].max() - 0.005
    return float(np.median(c["cand"][near]))          # middle of the near-optimal plateau

def est_at(t, ts, ns, pi0):
    m = len(ts); pred = int((ts >= t).sum()); fpr = (1 + (ns >= t).sum()) / (len(ns) + 1)
    D = (1 - pi0) * m; tp = float(np.clip(pred - pi0 * m * fpr, 0, D))
    return tp / max(pred, 1), tp / max(D, 1e-9), 2 * tp / max(pred + D, 1e-9), pred / m

THR, rows = {}, []
tot_tp = tot_pred = tot_D = 0.0
for cat in CATS:
    ns, ts, ss = E[cat]["loo"], E[cat]["test"], E[cat]["syn"]; m = len(ts)
    c = curve(ts, ns, CFG.STOREY_LAMBDA); pi0 = c["pi0"] if c else np.nan
    mode = CFG.THRESH_MODE
    if mode == "f1_estimate":
        if c is None or c["D"] < CFG.MIN_EST_DEFECTS or pi0 >= 0.97:
            print(f"[{cat}] F1 estimate unreliable (pi0={pi0:.2f}); falling back to normal-quantile rule.")
            mode = "quantile"
        else:
            bt = []
            for _ in range(CFG.N_BOOT):
                cb = curve(rng_b.choice(ts, m), rng_b.choice(ns, len(ns)), CFG.STOREY_LAMBDA)
                if cb is not None: bt.append(best_t(cb))
            t = float(np.median(bt)) if bt else best_t(c)
    if mode == "quantile": t = float(np.quantile(ns, CFG.NORMAL_Q))
    if mode == "fixed_z":  t = CFG.FIXED_Z
    THR[cat] = t
    if np.isfinite(pi0):
        pr, rc, f1, rate = est_at(t, ts, ns, pi0)
        tot_pred += (ts >= t).sum(); tot_D += (1 - pi0) * m; tot_tp += rc * (1 - pi0) * m
    else:
        pr = rc = f1 = rate = np.nan
    rows.append(dict(cat=cat, n_test=m, n_normal_ref=len(ns), pi0_est=pi0, est_defects=(1-pi0)*m if np.isfinite(pi0) else np.nan,
                     thr_z=t, mode=mode, pred_def_rate=(ts >= t).mean(), est_prec=pr, est_rec=rc, est_F1=f1,
                     normal_FPR_loo=(ns >= t).mean(), synthetic_recall=(ss >= t).mean()))
summary = pd.DataFrame(rows).set_index("cat").round(3)
print(summary.to_string())
if tot_pred > 0 and tot_D > 0:
    print(f"\nESTIMATED overall F1 (model-based, assumption-dependent, NOT a measured score): "
          f"{2*tot_tp/(tot_pred+tot_D):.3f}")

print("\nSensitivity: estimated F1 / predicted-defective rate if threshold = normal quantile q")
sens = {}
for cat in CATS:
    ns, ts = E[cat]["loo"], E[cat]["test"]; c = curve(ts, ns, CFG.STOREY_LAMBDA)
    if c is None: continue
    sens[cat] = {q: "%.2f / %.2f" % est_at(np.quantile(ns, q), ts, ns, c["pi0"])[2:] for q in (.80, .90, .95, .98, .99, 1.0)}
print(pd.DataFrame(sens).T.to_string())

In [ ]:
z = np.full(len(test_meta), np.nan); pred_def = np.zeros(len(test_meta), bool)
for cat in CATS:
    z[te_idx[cat]] = E[cat]["test"]; pred_def[te_idx[cat]] = E[cat]["test"] >= THR[cat]
assert not np.isnan(z).any()
# Competition labels are numeric: 0 = good/normal, 1 = defective.
labels = np.where(pred_def, 1, 0).astype(np.int64)

id_col, lab_col = sample_sub.columns[:2]
sub = sample_sub[[id_col]].merge(pd.DataFrame({id_col: test_meta["id"], lab_col: labels}), on=id_col, how="left")
os.makedirs("/kaggle/working", exist_ok=True)
sub.to_csv("/kaggle/working/submission.csv", index=False)
pd.DataFrame({"id": test_meta["id"], "category": test_meta["category"], "z": z, "pred": labels}) \
  .to_csv("/kaggle/working/test_scores_debug.csv", index=False)

# format checks
chk = pd.read_csv("/kaggle/working/submission.csv")
assert list(chk.columns) == [id_col, lab_col]
assert len(chk) == len(sample_sub) == len(test_df) and chk[id_col].is_unique
assert set(chk[id_col]) == set(test_df["id"]) and chk[lab_col].notna().all()
assert set(chk[lab_col].unique()).issubset({0, 1})
assert 0 < (chk[lab_col] == 1).sum() < len(chk), "degenerate: all one class"
print(chk[lab_col].value_counts()); print(chk.head())
print(pd.crosstab(test_meta["category"], labels))
print("Saved /kaggle/working/submission.csv")

In [ ]:
print(f"""
SUMMARY
- Detector: {', '.join(CFG.BACKBONES)} patch features (layers 2+3, {CFG.NEIGHBORHOOD}x{CFG.NEIGHBORHOOD} local pooling, dim {CFG.FEAT_DIM}), IMG_SIZE={CFG.IMG_SIZE}
- One kNN memory bank per category from ALL good train images (coreset ratio {CFG.CORESET_RATIO}); image score = {CFG.AGG}
- Calibration: leave-one-out normal scores, z-normalized per category; thresholds per category ({CFG.THRESH_MODE})
- Per-category thresholds (z): { {k: round(v, 2) for k, v in THR.items()} }
- Synthetic defects were used only for relative diagnostics; they do not set thresholds.
- Real defect-class F1 cannot be computed locally; only the Kaggle leaderboard measures it.
""")